# Paramétrage de l'environnement de travail et import des packages

In [7]:
import sys
from pathlib import Path

In [8]:
ROOT = Path.cwd().parents[0]

RAW_DATA = ROOT / "01_data" / "01_raw"
PROCESSED_DATA = ROOT / "01_data" / "02_processed"
MODEL_DATA = ROOT / "04_model"

%load_ext autoreload
%autoreload 2
sys.path.append(str(ROOT / "03_fonctions"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import pandas as pd
import streamlit as st
import shap
import joblib
import json
from huggingface_hub import InferenceClient
from fonctions_perso.machine_learning import BinaryMetricsSimple, graphique_courbe_pr, graphique_courbe_calibration, graphique_courbe_roc, ThresholdCostOptimizer
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

**Chargement des données/objets utiles**

À éxécuter une seule fois lors du démarrage

In [19]:
@st.cache_resource
def load_models_and_shap_values():

    modele_propre = joblib.load(MODEL_DATA / "fraud_detection_model_xgboost.joblib")
    modele_shap = joblib.load(MODEL_DATA / "fraud_detection_xgb_pas_calibre.joblib")
 
    return modele_propre, modele_shap

@st.cache_data
def load_sample_data():

    data_1_sample = joblib.load(PROCESSED_DATA / "1_fraud_data_sample.joblib")
    data_0_sample = joblib.load(PROCESSED_DATA / "0_fraud_data_sample.joblib")

    sample_data = pd.concat([data_1_sample,data_0_sample], axis = 0)
    sample_data = sample_data.sample(10)

    return sample_data

def score_transaction(input_dict):
    """
    input_dict : dict avec les features brutes (montant, type_magasin, heure, etc.)
    renvoie : probabilité de fraude + SHAP values locaux
    """
    X = pd.DataFrame([input_dict])
    proba = modele_propre.predict_proba(X)[:, 1][0]
    explainer = joblib.load(PROCESSED_DATA / "explainer.joblib")

    shap_values = explainer.shap_values(modele_shap.named_steps["preprocess"].transform(X))

    return X, proba, explainer, shap_values


def plot_local_shap(shap_values, X):
    """
    Affiche un waterfall / force plot ou bar plot des contributions SHAP pour une ligne.
    """
    shap.initjs()
    shap.force_plot(
        base_value=explainer.expected_value,
        shap_values=shap_values[0, :],
        features=X.iloc[0, :],
        matplotlib=True
    )
    st.pyplot(bbox_inches="tight", dpi=100)



def build_prompt_for_transaction(proba, input_dict, top_features):
    """
    proba: float (probabilité de fraude)
    input_dict: dict des features brutes de la transaction
    top_features: liste de tuples (feature_name, shap_value) triés par importance absolue
    """
    decision = "bloquée (suspecte)" if proba >= 0.226 else "acceptée"
    top_str = "\n".join(
        [f"- {name} (contribution SHAP = {value:+.3f})" for name, value in top_features]
    )

    prompt = f"""
Tu es un analyste fraude dans une banque française.
Explique en français, de façon claire et compréhensible pour un client,
pourquoi la transaction suivante est {decision} par le modèle de détection de fraude.

Contexte :
- Probabilité estimée de fraude : {proba:.3f}
- Décision automatique du modèle : {decision}

Caractéristiques de la transaction :
- Montant : {input_dict.get("montant_transaction")}
- Type de magasin : {input_dict.get("type_magasin")}
- Heure de la transaction : {input_dict.get("heure_transaction")}h
- État du client : {input_dict.get("etat_client")}
- Âge du client : {input_dict.get("age_client")}
- Distance domicile–magasin : {input_dict.get("distance_domicile_magasin")}

Principales variables ayant influencé le modèle (SHAP) :
{top_str}

Consignes :
- Adopte un ton professionnel mais pédagogique.
- Ne parle pas de “SHAP” ni de “modèle XGBoost”, parle de “algorithme interne de détection de fraude”.
- Mets en avant les éléments qui augmentent le risque et ceux qui le réduisent.
- Rédige 1 à 2 paragraphes maximum.
"""
    return prompt


hugging_face_json = "hugging_face_token.json"
with open(ROOT / hugging_face_json, "r") as f:
    secrets = json.load(f)

TOKEN = secrets["token_hugging_face"]

client = InferenceClient(
    model="mistralai/Mistral-7B-Instruct-v0.2",  
    token=TOKEN,
)


def call_llm_explanation(proba, shap_values, input_dict, feature_names, k_top=5):
    # shap_values: array 1D des contributions pour chaque feature (transaction unique)
    # feature_names: liste des noms de features alignés avec shap_values
    import numpy as np

    # top k features par importance absolue
    idx_sorted = np.argsort(-np.abs(shap_values))
    top_idx = idx_sorted[:k_top]
    top_features = [(feature_names[i], float(shap_values[i])) for i in top_idx]

    prompt = build_prompt_for_transaction(proba, input_dict, top_features)

    # appel Hugging Face
    output = client.text_generation(
        prompt,
        max_new_tokens=250,
        temperature=0.4,
        top_p=0.9,
        do_sample=True,
    )[0]["generated_text"]  # format de retour du client hfapi simple[web:281]

    return output


2026-03-07 18:52:33.126 No runtime found, using MemoryCacheStorageManager


**Onglet 1**

In [ ]:
def tab_ml_metrics(y_true_train, y_proba_train,
                   y_true_test, y_proba_test,
                   seuil_decision: float = 0.226):
    st.header("Métriques modèle (vue data scientist)")

    # 1. Prédictions binaires sur test avec ton seuil métier
    y_pred_test = (y_proba_test >= seuil_decision).astype(int)

    # 2. Utilisation de BinaryMetricsSimple
    metrics_obj = BinaryMetricsSimple(
        y_true=y_true_test,
        y_pred=y_pred_test,
        y_proba=y_proba_test
    )

    st.subheader("Métriques globales (test)")

    # DataFrame des métriques
    df_metrics = metrics_obj.get_metrics_df()
    st.dataframe(df_metrics)

    # Matrice de confusion (format texte que tu as défini)
    st.text(metrics_obj.print_confusion_matrix())

    # 3. Courbe Précision–Rappel (train vs test)
    st.subheader("Courbes Précision–Rappel (train vs test)")
    graphique_courbe_pr(
        y_train=y_true_train,
        train_proba=y_proba_train,
        y_test=y_true_test,
        test_proba=y_proba_test,
        figsize=(6, 5),
        save_path=None,
    )
    st.pyplot(plt.gcf())

    # 4. Courbe de calibration (train vs test)
    st.subheader("Courbes de calibration (train vs test)")
    graphique_courbe_calibration(
        y_train=y_true_train,
        train_proba=y_proba_train,
        y_test=y_true_test,
        test_proba=y_proba_test,
        n_bins=10,
        figsize=(6, 5),
        save_path=None,
    )
    st.pyplot(plt.gcf())


**Onglet 2**

In [ ]:
def tab_business_metrics(y_true_val, y_proba_val):
    st.header("Métriques métier & coût du seuil")

    # 1. Entrée des coûts métier
    col1, col2 = st.columns(2)
    with col1:
        cost_fp = st.number_input("Coût d’un faux positif (client embêté pour rien)", value=25, min_value=0)
    with col2:
        cost_fn = st.number_input("Coût d’une fraude non détectée", value=125, min_value=0)

    # 2. Fit de ton optimiseur sur l’échantillon de validation
    optimizer = ThresholdCostOptimizer(cost_fp=cost_fp, cost_fn=cost_fn, n_thresholds=200)
    optimizer.fit(y_true_val, y_proba_val)

    seuils = optimizer.seuils_
    costs = optimizer.costs_
    best_t = optimizer.get_best_threshold()
    best_cost = optimizer.get_best_cost()

    # 3. Slider Streamlit pour choisir un seuil
    st.subheader("Explorer le coût selon le seuil de décision")

    seuil_sel = st.slider(
        "Seuil de décision",
        float(seuils.min()),
        float(seuils.max()),
        float(best_t),
        step=float(seuils[1] - seuils[0])
    )

    # coût au seuil sélectionné
    idx_sel = int(np.argmin(np.abs(seuils - seuil_sel)))
    cost_sel = float(costs[idx_sel])

    # 4. Graphique Plotly interactif
    fig = go.Figure()

    # Courbe coût total
    fig.add_trace(
        go.Scatter(
            x=seuils,
            y=costs,
            mode="lines",
            name="Coût total",
            line=dict(color="#1f77b4")
        )
    )

    # Point seuil optimal (en rouge)
    fig.add_trace(
        go.Scatter(
            x=[best_t],
            y=[best_cost],
            mode="markers",
            name=f"Seuil optimal ({best_t:.3f})",
            marker=dict(color="red", size=10),
        )
    )

    # Point seuil sélectionné (en orange)
    fig.add_trace(
        go.Scatter(
            x=[seuil_sel],
            y=[cost_sel],
            mode="markers",
            name=f"Seuil sélectionné ({seuil_sel:.3f})",
            marker=dict(color="orange", size=10),
        )
    )

    # Ligne verticale sur le seuil sélectionné
    fig.add_vline(
        x=seuil_sel,
        line=dict(color="orange", dash="dash"),
        annotation_text=f"t={seuil_sel:.3f}\ncoût={cost_sel:,.0f}€",
        annotation_position="top right",
    )

    fig.update_layout(
        xaxis_title="Seuil de décision",
        yaxis_title="Coût total (FP/FN)",
        title="Coût total en fonction du seuil de décision",
        legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5),
    )

    st.plotly_chart(fig, use_container_width=True)

    # 5. Résumé texte métier
    st.markdown(
        f"""
        - **Seuil optimal (validation)** : {best_t:.3f}, coût minimum ≈ {best_cost:,.0f} €  
        - **Seuil sélectionné** : {seuil_sel:.3f}, coût associé ≈ {cost_sel:,.0f} €  

        Cela permet de voir comment le choix du seuil modifie directement le coût
        global en combinant faux positifs (clients embêtés pour rien) et fraudes non détectées.
        """
    )


**Onglet 3**

In [ ]:
def tab_transactions_and_explanations():
    st.header("Transactions et explications locales")

    st.markdown("Sélectionnez une transaction pour voir l’explication détaillée.")

    st.dataframe(df_sample)

    idx = st.number_input(
        "Indice de la transaction à expliquer",
        min_value=0,
        max_value=len(df_sample) - 1,
        value=0,
        step=1
    )

    row = df_sample.iloc[idx]
    st.write("Transaction sélectionnée :", row.to_dict())

    # scoring + SHAP local
    proba, shap_values, X = score_transaction(row.to_dict())
    st.write(f"Probabilité estimée de fraude : {proba:.3f}")

    st.subheader("Explication locale (SHAP)")
    plot_local_shap(shap_values, X)

    if st.button("Générer une explication métier (LLM)"):
        # extraire les top features SHAP, construire le prompt, etc.
        explanation = call_llm_explanation(proba, shap_values, row.to_dict())
        st.markdown(explanation)


**Onglet 4**

In [ ]:
def tab_simulator():
    st.header("Votre transaction serait-elle considérée comme frauduleuse ?")

    # quelques inputs utilisateur
    montant = st.number_input("Montant de la transaction", min_value=0.0, value=50.0)
    type_magasin = st.selectbox("Type de magasin", ["grocery_pos", "gas_transport", "shopping_pos", "misc_net"])
    heure = st.selectbox("Heure de la transaction", list(range(0, 24)))
    etat = st.selectbox("État du client", ["CA", "NY", "TX", "autre"])
    age = st.slider("Âge du client", 18, 90, 40)
    distance = st.number_input("Distance domicile–magasin (km)", min_value=0.0, value=5.0)

    if st.button("Évaluer la transaction"):
        input_dict = {
            "montant_transaction": montant,
            "type_magasin": type_magasin,
            "heure_transaction": heure,
            "etat_client": etat,
            "age_client": age,
            "distance_domicile_magasin": distance,
        }
        proba, shap_values, X = score_transaction(input_dict)
        st.write(f"Probabilité estimée de fraude : {proba:.3f}")

        decision = "ALERTE FRAUDE" if proba >= 0.226 else "ACCEPTÉE"
        st.write(f"Décision (seuil 0,226) : **{decision}**")

        st.subheader("Explication locale (SHAP)")
        plot_local_shap(shap_values, X)


**Main Streamlit**

In [ ]:
def main():
    st.set_page_config(page_title="Détection de fraude carte bancaire", layout="wide")

    st.title("Fraude carte bancaire – Modèle XGBoost explicable")

    tab1, tab2, tab3, tab4 = st.tabs([
        "Métriques ML",
        "Métriques métier & coûts",
        "Transactions & explications",
        "Simulateur de transaction",
    ])

    with tab1:
        tab_ml_metrics()
    with tab2:
        tab_business_metrics()
    with tab3:
        tab_transactions_and_explanations()
    with tab4:
        tab_simulator()


if __name__ == "__main__":
    main()
